# Generation for Charging Stations depending on relative Path Occupation

## Research Question

The feasibility of intelligently placing electrical charging stations on highly visited roads. SUMO and MatSIM deliver a full framework for simulating realistic traffic.


## Methods to determine traffic

- Occupation of lanes
- Color sections by own criteria, depending on traffic (manually, or by automation script) 
- Occupation of certain areas

## Deciding on Nomenclature

### Calculate with Occupation
ex: Lane A is occupied by 2 cars, X with 5s (seconds) and Y with 10s, in a timeframe of 60s. Car X enters on second 20-25, car Y enters on 23-33.
The occupancy of Lane A is a total of 13 seconds of 60. 
$$ ( 13 / 60 = 0,2167 ) $$

Mathematically, the lane is occupied 21.67% of the time.  

### Calculate with Density    
ex: Lane A (500m) is populated with 10 cars.
$$ 10 (cars) / 0.5 (km) =  20  cars / km $$ 

With lots of traffic the value may approach a value of around 100 cars/km.

## Chosen method to determine traffic

The problem requires a different solution than the occupation of a lane, because a high occupation does not necessarily hint towards a lot of traffic. So, the initial method will be utilizing the density of each lane across the whole simulation time span.

Density allows to track every lane independently and extract information for these lanes. For now Occupation is less of a prioritiy.

## Design of method

The first step to complete the goal of finding a way to calculate beneficial positions for charging stations, is to extract the density and the lane's id, to which it belongs. These two values do not solve the problem itself, but deliver the information needed for setting up the criteria and a fitting algorithm.  


## Implementation of method

Below is the first part of code, that allows the extraction of the density and lane id into two variables.

In [23]:
import xml.etree.ElementTree as ET

tree = ET.parse('welzheim_edges.xml')
root = tree.getroot()

density_values = []
lane_ids = []
for edge in root.findall('.//edge'):
    density = edge.get('density')
    lane_ids.append(edge.get('id'))
    if density:
        density_values.append(density)

for i in range(len(density_values)):
    if (density_values[i] > "2"):
        print("density (lane " + str(i) + "): " + str(density_values[i]))
print("lane_ids:" + str(lane_ids))

density (lane 122): 2.06
density (lane 154): 2.82
density (lane 155): 3.60
density (lane 223): 3.37
density (lane 225): 4.59
density (lane 374): 3.15
density (lane 455): 2.12
density (lane 456): 2.16
density (lane 498): 2.33
density (lane 510): 3.85
density (lane 656): 5.64
density (lane 657): 4.36
density (lane 742): 2.15
density (lane 870): 4.48
density (lane 871): 4.77
lane_ids:['1', '10', '100', '1000', '1001', '1002', '1003', '1004', '1005', '1007', '1008', '1009', '101', '1010', '1011', '1014', '1015', '1016', '1017', '1018', '1019', '102', '1020', '1021', '1022', '1023', '1024', '1025', '1026', '1027', '1028', '1029', '103', '1030', '1031', '1033', '1034', '1035', '1038', '1039', '104', '1040', '1041', '1042', '1043', '1044', '1045', '1046', '1047', '1048', '1049', '105', '1050', '1051', '1052', '1053', '1055', '1056', '1057', '1058', '1059', '106', '1060', '1061', '1062', '1063', '1064', '1065', '1066', '1067', '1068', '1069', '107', '1070', '1071', '1072', '1073', '1074', '107

## Algorithm to determine positioning of charging stations

The algorithm will take into account which lanes are busy and will put electrical charging stations depending on their traffic density.

## Requirements Engineering

It does not make sense to put charging stations everywhere, where there is a density value higher than X, assuming some streets might be overcrowded and others void of charging stations. So, a few criterias need to be elaborated to guarantee and efficient system to distribute charging stations.

In [ ]:
sumo_cfg = 'welzheim.sumocfg'
network_cfg = ET.parse(sumo_cfg)
root_cfg = network_cfg.getroot()

print(root_cfg)

if (root.find('.//additional-files') is None):
    additional = ET.Element('additional-files')
    additional.set('value', 'welzheim_edges.xml')
    root_cfg.append(additional)


network_cfg.write(sumo_cfg, encoding='utf-8', xml_declaration=True)


<Element 'configuration' at 0x000002BC292722A0>


In [ ]:
import os

filename = "welzheim.add.xml"

if not os.path.exists(filename):
    xml_content = """<?xml version="1.0" encoding="UTF-8"?>\n<additional>\n</additional>"""
    with open(filename, "w", encoding="utf-8") as file:
        file.write(xml_content)
    
    print(f"Datei '{filename}' wurde erstellt.")
else:
    print(f"Datei '{filename}' existiert bereits.")


Datei 'welzheim.add.xml' wurde erstellt.
